# Indexing TREC Robust 2005 by OpenSearch for BM25 Model

- [aquaint/trec-robust-2005](https://ir-datasets.com/aquaint.html#aquaint/trec-robust-2005)

### Install python modules

In [1]:
import sys
!{sys.executable} -m pip install ir_datasets pandas opensearch-py

### Load helper modules

In [2]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [3]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'qIJ28Ej2TCC6_LI88zZLlw',
 'name': '4db878c40bab',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2026-02-07T07:54:31.169913465Z',
             'build_hash': 'bbc94f0bdc3a759011e6529ecfe52840856f91a3',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.2',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.5.0'}}


### Index a Corpus for BM25 Model

In [7]:
import ir_datasets
dataset_name = "aquaint/trec-robust-2005"
dataset = ir_datasets.load(dataset_name)
docstore = dataset.docs_store()
docstore.build()

Index structure

In [5]:
index_name = "trec_robust_2005_bm25"
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

In [6]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}
response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

{'acknowledged': True,
 'index': 'trec_robust_2005_bm25',
 'shards_acknowledged': True}


Indexing

In [8]:
from bs4 import BeautifulSoup
def parse_marked_up_doc(marked_up_doc):
    # Parse the content using BeautifulSoup
    soup = BeautifulSoup(marked_up_doc, 'html.parser')
    
    # Extract the title from the <HEADLINE> tag
    headline_tag = soup.find('headline')
    title = headline_tag.get_text(strip=True) if headline_tag else "No Title"
    
    # Extract the text from the <TEXT> tag and join paragraphs
    text = ' '.join(p.get_text(strip=True) for p in soup.find_all('p'))
    
    return title, text

In [9]:
def prepare_documents(dataset, docstore):
    """
    Prepare individual documents for indexing, with progress tracking.
    """
    total_docs = sum(1 for _ in dataset.docs_iter())  # Count the total documents
    progress = tqdm(total=total_docs, desc="Indexing Documents")  # Progress bar

    for doc in dataset.docs_iter():
        # Parse the document
        title, text = parse_marked_up_doc(docstore.get(doc.doc_id).marked_up_doc)
        text = text.replace("\n", " ")
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "text": text,
                "title": title
            }
        }
        progress.update(1)  # Update progress bar

    progress.close()  # Close the progress bar

In [10]:
from opensearchpy.helpers import bulk
success, failed = bulk(client, prepare_documents(dataset, docstore), index=index_name)

Indexing Documents: 100%|██████████| 1033461/1033461 [07:16<00:00, 2369.62it/s]


#### Search Test

In [11]:
def search(query: str, size: int = 10) -> dict:
    body = {
        "size": size,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^2", "text"] # title gets a boost
            }
        },
    }

    return client.search(index=index_name, body=body)

In [ ]:
q = "killer bee attack human"
resp = search(q, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for query: {q}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")